# **응급상황 자동 인식 및 응급실 연계 서비스**
# **단계4 : 통합 - pipeline**

## **0.미션**

단계 4에서는, 단계1,2,3 에서 생성한 함수들을 모듈화하고, 단위 테스트 및 파이프라인 코드를 작성합니다.

* **미션6**
    * 단위 테스트
        * 각 기능(함수)에 대해 단계별로 테스트를 수행하며 오류를 해결합니다.
    * 파이프라인 구축
        * 단계1의 결과가 단계2 모델에 input이 되고, 모델의 예측 결과를 기반으로
        * 응급실 추천되도록
        * 조원들이 녹음한 음성 파일에 임의의 좌표(위도, 경도)값을 부여
            * 음성파일 이름과 좌표를 저장하는 별도 데이터셋 생성
        * 각 모듈을 연결하여 파이프라인 구성하는 ipynb 파일 생성



## **1.환경설정**

### (1) 경로 설정

구글 드라이브 연결

#### 1) 구글 드라이브 폴더 생성
* 새 폴더(project6_2)를 생성하고
* 제공 받은 파일을 업로드

#### 2) 구글 드라이브 연결

In [15]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [16]:
path = '/content/drive/MyDrive/mini-project-6-1/' ## 수정 필요

### (2) 라이브러리

#### 1) 필요한 라이브러리 설치

* requirements.txt 파일의 [경로 복사]를 한 후,
* 아래 경로에 붙여 넣기

In [17]:
# 경로 : /content/drive/MyDrive/project6_2/requirements.txt
# 경로가 다른 경우 아래 코드의 경로 부분을 수정하세요.

!pip install -r /content/drive/MyDrive/mini-project-6-1/requirements.txt

#### 2) 라이브러리 로딩

In [18]:
!ls {path}

'1. 음성 인식 및 요약_1일차 종합 1.ipynb'   emergency2.py
 1_음성_인식_및_요약_DH.ipynb		    emergency.py
'1. 음성 인식 및 요약.ipynb'		   'emergency_room_1일차 종합.csv'
'2. 응급 등급 분류.ipynb'		    emotion.csv
'2. 응급 등급 분류 - 김찬주.ipynb'	    fine_tuned_bert
'3. 응급실 연계(추천).ipynb'		    ktas.csv
'3. 응급실 연계(추천)_준혁.ipynb'	    latitude_longitude.csv
'4-1. 모듈화.ipynb'			    map_key.txt
'4-2. 통합.ipynb'			    __pycache__
 api_key.txt				    requirements.txt
 audio					    self_audio
 audio_location.xlsx			   '예제_BERT_fine tuning_emotion.ipynb'
 dataset100.csv				   '응급실 정보.csv'
 dataset.csv				   '중증도 카테고리.csv'
 dataset_v1.csv				   '참조_API사용_ChatGPT API.ipynb'


In [19]:
#필요한 라이브러리 설치 및 불러우기
import os
import pandas as pd
import numpy as np

from haversine import haversine
import requests
import json

# 더 필요한 라이브러리 추가 -------------
import sys
sys.path.append(path)
import xml.etree.ElementTree as ET
import matplotlib.pyplot as plt
import openai
from openai import OpenAI
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
from geopy.distance import geodesic
from geopy import Point
from haversine import haversine,haversine_vector, Unit
from warnings import filterwarnings
FutureWarning
filterwarnings('ignore')

# 조에서 생성한 모듈 불러오기 -------------
from emergency import RecommendHospital3

In [22]:
filename = "5-1.m4a"  # 예제 파일명
filepath = 'map_key.txt'    
with open(filepath, 'r') as file:
    map_key = json.load(file)
    naver_id, naver_key = map_key['c_id'], map_key['c_key']
path = "/content/drive/MyDrive/mini-project-6-1/"  # 파일 경로

## **2. 단위 테스트**

* 세부사항 : 아래 단계별로 데이터가 순차적으로 처리되도록 단위 테스트를 진행합니다.

In [23]:
hospital_recommender = RecommendHospital3(filename, naver_id, naver_key, path)

### (1) open ai key 등록

In [8]:
hospital_recommender.load_api_key()

### (2) audio to text to summary

In [9]:
hospital_recommender.audio_summary()

지금 편방동에 있는 집입니다. 며칠 전부터 감기 증상을 보이던 가족이 지금도 가벼운 기침과 콧물을 흘리고 있습니다. 응급상황은 아니지만 약국이 문을 닫았고 병원 진료를 받고 싶습니다. 구급차 요청드리지만 급하지 않으니 천천히 오셔도 됩니다.
 가족이 가벼운 기침과 콧물 증상이 지속되고 있습니다. 응급상황은 아니지만 약국이 문을 닫았고 병원 진료를 받고 싶다고 합니다. 구급차 요청을 해주셨지만 급점하지 않으니 천천히 오셔도 된다고 합니다.


('가족이 가벼운 기침과 콧물 증상이 지속되고 있습니다. 응급상황은 아니지만 약국이 문을 닫았고 병원 진료를 받고 싶다고 합니다. 구급차 요청을 해주셨지만 급점하지 않으니 천천히 오셔도 된다고 합니다.',
 37.5384352452626,
 126.989828026954)

### (3) 응급실 등급분류

In [10]:
hospital_recommender.classify_situation()

지금 편방동에 있는 집입니다. 며칠 전부터 감기 증상을 보이던 가족이 지금도 가벼운 기침과 콧물을 흘리고 있습니다. 응급상황은 아니지만 약국이 문을 닫았고 병원 진료를 받고 싶습니다. 구급차 요청드리지만 급하지 않으니 천천히 오셔도 됩니다.
 가벼운 기침과 콧물로 인한 약국 진료가 필요하다고 판단됩니다. 응급상황은 아닙니다.
4 등급


(4, 37.5384352452626, 126.989828026954)

### (4) 응급실추천

In [24]:
hospital_recommender.search_map()

지금 편방동에 있는 집입니다. 며칠 전부터 감기 증상을 보이던 가족이 지금도 가벼운 기침과 콧물을 흘리고 있습니다. 응급상황은 아니지만 약국이 문을 닫았고 병원 진료를 받고 싶습니다. 구급차 요청드리지만 급하지 않으니 천천히 오셔도 됩니다.
 감기 증상으로 가벼운 기침과 콧물을 보이고 있는 상황입니다. 신고자는 응급상황은 아니지만 병원 진료를 받고 싶어하며, 구급차를 요청 중이지만 급하지 않다고 합니다. 

따라서, 상황을 종합해 볼 때 응급상황은 아닌 것으로 판단됩니다.
4 등급


'가까운 병원을 찾아가는 것을 추천드립니다.'

## **3. 파이프라인**

* 세부사항
    * [2. 단계별 테스트] 의 내용을 순차적으로 정리합니다.
        * 데이터 처리 전 준비작업 : 한번 실행하면 되는 영역
            * 키, 데이터로딩
            * 모델/토크나이저 로딩
        * 입력값이 들어 왔을 때 출력값까지 처리되는 영역

In [12]:
filename = "1-1.m4a"  # 예제 파일명
filepath = 'map_key.txt'    
with open(filepath, 'r') as file:
    map_key = json.load(file)
    naver_id, naver_key = map_key['c_id'], map_key['c_key']
path = "/content/drive/MyDrive/mini-project-6-1/"  # 파일 경로

In [13]:
hospital_recommender = RecommendHospital3(filename, naver_id, naver_key, path)
result = hospital_recommender.search_map()
print(result)

신동현대아파트입니다. 제 아버지가 갑자기 가슴을 움켜주더니 의식을 잃고 바닥에 쓰러졌습니다. 호흡이 멈춘 상태고 제가 지금 심폐소생술을 시도하고 있습니다. 세세동기같은 응급장비가 필요할 것 같습니다. 빨리 구급차 보내주세요. 의식이 돌아오지 않고 있습니다.
 의식을 잃고 호흡이 멈춘 상태이며 심폐소생술을 시도 중이고, 세세동기 같은 응급장비가 필요하다고 판단됩니다. 구급차를 신속히 호출해야 합니다.
1 등급
{'목적지': ['분당서울대학교병원', '대진의료재단분당제생병원', '국군수도병원', '차의과학대학교분당차병원', '성모윌병원'], '경과시간': ['0시간 8분', '0시간 12분', '0시간 14분', '0시간 16분', '0시간 20분'], '거리': [2.626, 4.707, 5.973, 6.892, 10.269], '택시비+톨비': [5500, 7300, 8300, 9300, 12100]}


In [14]:
pd.DataFrame(result)

,목적지,경과시간,거리,택시비+톨비
0,분당서울대학교병원,0시간 8분,2.626,5500
1,대진의료재단분당제생병원,0시간 12분,4.707,7300
2,국군수도병원,0시간 14분,5.973,8300
3,차의과학대학교분당차병원,0시간 16분,6.892,9300
4,성모윌병원,0시간 20분,10.269,12100
